<a href="https://colab.research.google.com/github/Cristiand056/proyecto_sena_chatbot/blob/main/proyecto_sena_chatbot_get_response_prov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from tensorflow.keras.models import load_model
import nltk
from nltk.stem import SnowballStemmer
import numpy as np
import pickle
import json
import random

In [5]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [6]:
stemmer    = SnowballStemmer('spanish')
modelo     = load_model('model/chatbot_model.h5')
words      = pickle.load(open('model/words.pkl', 'rb'))
classes    = pickle.load(open('model/classes.pkl', 'rb'))
intents    = json.load(open('model/intents.json', encoding='utf-8'))

UMBRAL_CONFIANZA = 0.75  # mínimo para dar una respuesta

def bag_of_words(frase: str) -> np.ndarray:
    # Tokeniza y lematiza el mensaje del usuario
    tokens = nltk.word_tokenize(frase.lower(), language='spanish')
    tokens_stem = [stemmer.stem(t) for t in tokens]
    bag = [1 if w in tokens_stem else 0 for w in words]
    return np.array(bag)

def predecir_intencion(frase: str) -> tuple[str, float]:
    # Convierte el mensaje en BoW y predice la intención
    bow = bag_of_words(frase)
    resultado = modelo.predict(np.array([bow]), verbose=0)[0]
    confianza = float(np.max(resultado))
    tag = classes[np.argmax(resultado)]
    return tag, confianza

def obtener_respuesta(frase: str) -> str:
    tag, confianza = predecir_intencion(frase)

    if confianza < UMBRAL_CONFIANZA:
        return "No entendí bien tu consulta. ¿Puedes reformularla o contactarnos directamente?"

    for intent in intents['intents']:
        if intent['tag'] == tag:
            return random.choice(intent['responses'])

    return "Ocurrió un error. Por favor contáctanos por WhatsApp."

## **Prueba**

In [7]:
# Frases que SÍ están en intents (debería acertar)
print(obtener_respuesta("tienen ron"))
print(obtener_respuesta("hola"))

# Frases que NO están pero son similares (aquí se ve la calidad real)
print(obtener_respuesta("me consiguen una botella de guaro"))
print(obtener_respuesta("quiero pedir unas cosas"))
print(obtener_respuesta("a qué horas abren"))
print(obtener_respuesta("se me olvidó la clave"))
print(obtener_respuesta("cuánto vale el marlboro"))

Manejamos licores nacionales e importados. ¿Qué referencia buscas?
¡Hola! Bienvenido a Distribuidora X, ¿en qué te puedo ayudar hoy?
Tenemos una amplia selección de licores. ¿Tienes alguna marca en específico?
Puedes crear tu pedido desde la sección Pedidos del menú principal.
Nuestro horario es de lunes a viernes de 8am a 6pm y sábados de 8am a 2pm.
Si no recibes el correo de recuperación, contáctanos directamente y te ayudamos a restablecer el acceso.
Contáctanos por WhatsApp y con gusto te enviamos la lista de precios actualizada.
